In [1]:

# CELL 1 — IMPORTS
import os, re, json, math, time, gc, copy, random, warnings, subprocess, sys
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import scipy
from scipy import signal, linalg, stats
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score,
    confusion_matrix, classification_report, precision_recall_fscore_support
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    import mne
    from mne import create_info
    from mne.io import read_raw_gdf
except Exception:
    mne = None

try:
    import optuna
except Exception:
    optuna = None

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True, precision=4)
pd.set_option("display.max_columns", 100)


In [8]:

# CELL 2 — CONFIGURATION
@dataclass
class CFG:
    # Dataset
    data_root: str = "/Users/ashokvarmabevara/Project2/BCI IV-2a"
    cache_dir: str = "./cache_bnci2a_hybrid"
    results_dir: str = "./results_bnci2a_hybrid"
    checkpoint_dir: str = "./checkpoints_bnci2a_hybrid"
    n_channels: int = 22
    sfreq: int = 250
    n_times: int = 1000
    n_classes: int = 4
    class_names: Tuple[str, ...] = ("LeftHand", "RightHand", "Feet", "Tongue")
    event_codes: Tuple[int, ...] = (769, 770, 771, 772)

    # Main protocol
    protocol: str = "strict_loso"  # strict_loso | target_adaptation
    preprocess_mode: str = "nf_raw"  # nf_raw | paper_1_38 | paper_8_30
    use_car: bool = True
    normalize: str = "train_zscore_global"  # train_zscore_global | train_zscore_channel
    validation_subjects_per_fold: int = 2
    inner_seed: int = 2026

    # Architecture switches
    use_nf_eeg: bool = True
    use_eegnet: bool = True
    use_multiscale: bool = True
    use_transformer: bool = True
    use_bilstm: bool = True
    use_channel_attention: bool = True
    attention_type: str = "se"  # se | cbam | gate | none
    use_domain_alignment: bool = False
    domain_alignment: str = "mmd"  # mmd
    use_center_loss: bool = False
    use_supcon: bool = False
    use_fbgan: bool = False  # intentionally disabled by default; edit to True
    use_target_adaptation: bool = False
    use_ensemble: bool = False
    strict_source_whitening: bool = False
    target_use_ea: bool = False

    # NF-EEG tokenizer
    patch_spatial: int = 4
    patch_temporal: int = 25
    patch_stride_spatial: int = 4
    patch_stride_temporal: int = 25
    token_dim: int = 128

    # Multi-scale CNN
    multiscale_channels: int = 64
    multiscale_kernels: Tuple[int, ...] = (3, 5, 7, 11, 15)
    dropout: float = 0.35
    activation: str = "gelu"

    # EEGNet
    eegnet_f1: int = 8
    eegnet_d: int = 2
    eegnet_f2: int = 16
    eegnet_kernel: int = 64

    # Transformer
    transformer_dim: int = 128
    transformer_heads: int = 4
    transformer_layers: int = 4
    transformer_ff: int = 256
    transformer_dropout: float = 0.2
    max_tokens: int = 200

    # BiLSTM
    bilstm_hidden: int = 128
    bilstm_layers: int = 2
    bilstm_dropout: float = 0.25

    # Fusion
    fusion_dim: int = 128

    # Optimization
    batch_size: int = 32
    grad_accum_steps: int = 1
    epochs: int = 100
    patience: int = 15
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    warmup_epochs: int = 5
    use_amp: bool = False
    num_workers: int = 0

    # Losses
    center_lambda: float = 0.05
    supcon_lambda: float = 0.05
    domain_lambda: float = 0.02
    center_lr: float = 0.5

    # Target adaptation / FBGAN
    target_calibration_trials_per_class: int = 12
    fbgan_latent_dim: int = 1600
    fbgan_lr: float = 1e-4
    fbgan_batch_size: int = 5
    fbgan_epochs: int = 100
    fbgan_samples_per_class: int = 750
    fbgan_aug_ratio: float = 0.30

    # Evaluation / search
    seed: int = 20260824
    fast_dev_run: bool = False
    run_only_subjects: Optional[List[str]] = None

CFG_DEFAULT = CFG()
print(asdict(CFG_DEFAULT))


{'data_root': '/Users/ashokvarmabevara/Project2/BCI IV-2a', 'cache_dir': './cache_bnci2a_hybrid', 'results_dir': './results_bnci2a_hybrid', 'checkpoint_dir': './checkpoints_bnci2a_hybrid', 'n_channels': 22, 'sfreq': 250, 'n_times': 1000, 'n_classes': 4, 'class_names': ('LeftHand', 'RightHand', 'Feet', 'Tongue'), 'event_codes': (769, 770, 771, 772), 'protocol': 'strict_loso', 'preprocess_mode': 'nf_raw', 'use_car': True, 'normalize': 'train_zscore_global', 'validation_subjects_per_fold': 2, 'inner_seed': 2026, 'use_nf_eeg': True, 'use_eegnet': True, 'use_multiscale': True, 'use_transformer': True, 'use_bilstm': True, 'use_channel_attention': True, 'attention_type': 'se', 'use_domain_alignment': False, 'domain_alignment': 'mmd', 'use_center_loss': False, 'use_supcon': False, 'use_fbgan': False, 'use_target_adaptation': False, 'use_ensemble': False, 'strict_source_whitening': False, 'target_use_ea': False, 'patch_spatial': 4, 'patch_temporal': 25, 'patch_stride_spatial': 4, 'patch_stride_

In [9]:

# CELL 3 — REPRODUCIBILITY + DEVICE
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass

set_seed(CFG_DEFAULT.seed)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Torch:", torch.__version__)
print("Device:", DEVICE)
print("MPS:", torch.backends.mps.is_available())
print("CUDA:", torch.cuda.is_available())

Path(CFG_DEFAULT.cache_dir).mkdir(parents=True, exist_ok=True)
Path(CFG_DEFAULT.results_dir).mkdir(parents=True, exist_ok=True)
Path(CFG_DEFAULT.checkpoint_dir).mkdir(parents=True, exist_ok=True)


Torch: 2.10.0
Device: mps
MPS: True
CUDA: False


In [10]:

# CELL 4 — DATASET DISCOVERY
def discover_dataset_files(root: str) -> Dict[str, List[str]]:
    root = str(Path(root).expanduser())
    if not os.path.exists(root):
        raise FileNotFoundError(
            f"DATA_ROOT does not exist: {root}\n"
            "Edit CFG_DEFAULT.data_root in CELL 2."
        )
    gdf = sorted(str(p) for p in Path(root).rglob("*.gdf"))
    npz = sorted(str(p) for p in Path(root).rglob("*.npz"))
    pkl = sorted(str(p) for p in Path(root).rglob("*.pkl"))
    return {"gdf": gdf, "npz": npz, "pkl": pkl}

files = discover_dataset_files(CFG_DEFAULT.data_root)
print({k: len(v) for k, v in files.items()})
print("Example files:")
for k, v in files.items():
    print(k, v[:5])


{'gdf': 18, 'npz': 0, 'pkl': 0}
Example files:
gdf ['/Users/ashokvarmabevara/Project2/BCI IV-2a/A01E.gdf', '/Users/ashokvarmabevara/Project2/BCI IV-2a/A01T.gdf', '/Users/ashokvarmabevara/Project2/BCI IV-2a/A02E.gdf', '/Users/ashokvarmabevara/Project2/BCI IV-2a/A02T.gdf', '/Users/ashokvarmabevara/Project2/BCI IV-2a/A03E.gdf']
npz []
pkl []


In [11]:

# CELL 5 — EEG LOADING UTILITIES
CLASS_NAME_TO_ID = {name: i for i, name in enumerate(CFG_DEFAULT.class_names)}
EVENT_TO_CLASS = dict(zip(CFG_DEFAULT.event_codes, range(CFG_DEFAULT.n_classes)))

def infer_subject_from_path(path: str) -> str:
    m = re.search(r"(A\d{2})", Path(path).name.upper())
    return m.group(1) if m else Path(path).stem.upper()

def _pick_eeg_channels(raw):
    eeg_picks = mne.pick_types(raw.info, eeg=True, eog=False, stim=False, exclude="bads")
    if len(eeg_picks) < CFG_DEFAULT.n_channels:
        raise RuntimeError(f"Found only {len(eeg_picks)} EEG channels in {raw.filenames[0]}")
    return eeg_picks[:CFG_DEFAULT.n_channels]

def _annotation_event_map(raw):
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    reverse = {}
    for name, code in event_id.items():
        digits = re.findall(r"\d+", str(name))
        if digits:
            reverse[code] = int(digits[-1])
    return events, reverse

def load_gdf_trials(path: str, cfg: CFG_DEFAULT):
    if mne is None:
        raise ImportError("mne is required for GDF loading.")
    raw = read_raw_gdf(path, preload=True, verbose=False)
    picks = _pick_eeg_channels(raw)

    if cfg.use_car:
        raw.pick(picks)
        raw.set_eeg_reference("average", projection=False, verbose=False)
    else:
        raw.pick(picks)

    events, reverse = _annotation_event_map(raw)
    rows = []
    for ev in events:
        onset, _, event_code_internal = ev
        true_code = reverse.get(event_code_internal, event_code_internal)
        if true_code in EVENT_TO_CLASS:
            rows.append((onset, EVENT_TO_CLASS[true_code]))

    n_samples = int(round(cfg.n_times))
    X_list, y_list = [], []
    for onset_sample, label in rows:
        start = int(onset_sample)
        stop = start + n_samples
        if stop <= raw.n_times:
            x = raw.get_data(start=start, stop=stop) * 1e6  # µV
            if x.shape == (cfg.n_channels, cfg.n_times):
                X_list.append(x.astype(np.float32))
                y_list.append(label)

    return (
        np.stack(X_list) if X_list else np.empty((0, cfg.n_channels, cfg.n_times), np.float32),
        np.asarray(y_list, np.int64),
        infer_subject_from_path(path)
    )


In [16]:
# ============================================================
# CELL 6 — BUILD DATASET — FINAL FIX
# ============================================================

def normalize_subjects(subjects_raw, n_samples, fallback_subject):

    if subjects_raw is None:
        return np.full(
            n_samples,
            str(fallback_subject),
            dtype=str
        )

    arr = np.asarray(subjects_raw)

    # Scalar subject: "A01"
    if arr.ndim == 0:
        return np.full(
            n_samples,
            str(arr.item()),
            dtype=str
        )

    arr = arr.reshape(-1)

    # One subject ID for the whole file
    if len(arr) == 1:
        return np.full(
            n_samples,
            str(arr[0]),
            dtype=str
        )

    # One subject ID per trial
    if len(arr) == n_samples:
        return arr.astype(str)

    raise ValueError(
        f"Subject metadata mismatch: "
        f"{len(arr)} IDs for {n_samples} trials."
    )


def load_npz(path):

    d = np.load(
        path,
        allow_pickle=True
    )

    # --------------------------------------------------------
    # Find EEG data
    # --------------------------------------------------------
    x_key = next(
        (
            k for k in
            ["X", "x", "data", "signals"]
            if k in d.files
        ),
        None
    )

    # --------------------------------------------------------
    # Find labels
    # --------------------------------------------------------
    y_key = next(
        (
            k for k in
            ["y", "Y", "labels"]
            if k in d.files
        ),
        None
    )

    if x_key is None:
        raise ValueError(
            f"No X array found in {path}\n"
            f"Available keys: {d.files}"
        )

    if y_key is None:
        raise ValueError(
            f"No y array found in {path}\n"
            f"Available keys: {d.files}"
        )

    X = np.asarray(
        d[x_key],
        dtype=np.float32
    )

    y = np.asarray(
        d[y_key]
    ).reshape(-1)

    if X.ndim != 3:
        raise ValueError(
            f"Expected X=(N,C,T), got {X.shape}"
        )

    if len(X) != len(y):
        raise ValueError(
            f"X/y mismatch: "
            f"X={len(X)}, y={len(y)}"
        )

    # --------------------------------------------------------
    # Convert labels
    # --------------------------------------------------------
    if y.dtype.kind in {"U", "S", "O"}:

        label_map = {
            "left": 0,
            "left hand": 0,
            "lh": 0,

            "right": 1,
            "right hand": 1,
            "rh": 1,

            "feet": 2,
            "both feet": 2,
            "foot": 2,

            "tongue": 3,
        }

        converted = []

        for value in y:

            value = str(value).strip().lower()

            if value in label_map:
                converted.append(
                    label_map[value]
                )
            else:
                converted.append(
                    int(float(value))
                )

        y = np.asarray(
            converted,
            dtype=np.int64
        )

    else:

        y = y.astype(
            np.int64
        )

        # 1,2,3,4 -> 0,1,2,3
        if set(
            np.unique(y).tolist()
        ) == {1, 2, 3, 4}:

            y = y - 1

    # --------------------------------------------------------
    # Subject information
    # --------------------------------------------------------
    subject_key = next(
        (
            k for k in
            [
                "subjects",
                "subject",
                "groups",
                "subject_id"
            ]
            if k in d.files
        ),
        None
    )

    if subject_key is not None:
        subjects_raw = d[subject_key]
    else:
        subjects_raw = None

    fallback_subject = infer_subject_from_path(
        path
    )

    subjects = normalize_subjects(
        subjects_raw,
        len(X),
        fallback_subject
    )

    return (
        X,
        y.astype(np.int64),
        subjects.astype(str)
    )


def build_dataset(cfg):

    X_parts = []
    y_parts = []
    subject_parts = []

    # ========================================================
    # LOAD NPZ
    # ========================================================

    if len(files["npz"]) > 0:

        print(
            f"Found {len(files['npz'])} NPZ files."
        )

        for path in files["npz"]:

            print("\nLoading:")
            print(path)

            Xi, yi, si = load_npz(path)

            print(
                "  X         :",
                Xi.shape
            )

            print(
                "  y         :",
                yi.shape
            )

            print(
                "  subjects  :",
                si.shape
            )

            print(
                "  subject IDs:",
                np.unique(si).tolist()
            )

            X_parts.append(Xi)
            y_parts.append(yi)
            subject_parts.append(si)

    # ========================================================
    # LOAD GDF
    # ========================================================

    elif len(files["gdf"]) > 0:

        print(
            f"Found {len(files['gdf'])} GDF files."
        )

        for path in files["gdf"]:

            print("\nLoading:")
            print(path)

            Xi, yi, sid = load_gdf_trials(
                path,
                cfg
            )

            if len(yi) == 0:

                print(
                    "  WARNING: no valid trials"
                )

                continue

            si = np.full(
                len(yi),
                str(sid),
                dtype=str
            )

            print(
                "  X         :",
                Xi.shape
            )

            print(
                "  y         :",
                yi.shape
            )

            print(
                "  subjects  :",
                si.shape
            )

            print(
                "  subject    :",
                sid
            )

            X_parts.append(Xi)
            y_parts.append(
                yi.astype(np.int64)
            )
            subject_parts.append(si)

    else:

        raise FileNotFoundError(
            "\nNo .gdf or .npz files found.\n"
            f"Data root: {cfg.data_root}"
        )

    # ========================================================
    # CHECK
    # ========================================================

    if len(X_parts) == 0:

        raise RuntimeError(
            "No EEG trials were loaded."
        )

    # ========================================================
    # CONCATENATE
    # ========================================================

    X = np.concatenate(
        X_parts,
        axis=0
    )

    y = np.concatenate(
        y_parts,
        axis=0
    )

    subjects = np.concatenate(
        subject_parts,
        axis=0
    )

    print("\n" + "=" * 70)
    print("BEFORE SHAPE FILTERING")
    print("=" * 70)

    print(
        "X       :",
        X.shape
    )

    print(
        "y       :",
        y.shape
    )

    print(
        "subjects:",
        subjects.shape
    )

    # ========================================================
    # CRITICAL VALIDATION
    # ========================================================

    if len(X) != len(y):

        raise RuntimeError(
            f"X/y mismatch: "
            f"{len(X)} vs {len(y)}"
        )

    if len(X) != len(subjects):

        raise RuntimeError(
            f"X/subjects mismatch: "
            f"{len(X)} vs {len(subjects)}"
        )

    # ========================================================
    # SHAPE FILTER
    # ========================================================

    keep = (
        (X.shape[1] == cfg.n_channels)
        &
        (X.shape[2] == cfg.n_times)
    )

    removed = int(
        np.sum(~keep)
    )

    if removed > 0:

        print(
            f"Removing {removed} "
            "invalid trials."
        )

        X = X[keep]
        y = y[keep]
        subjects = subjects[keep]

    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    if len(X) != len(y):

        raise RuntimeError(
            "Final X/y mismatch."
        )

    if len(X) != len(subjects):

        raise RuntimeError(
            "Final X/subjects mismatch."
        )

    # ========================================================
    # IMPORTANT:
    # DO NOT SORT THE DATASET
    # ========================================================
    #
    # Sorting is unnecessary for LOSO.
    # SUBJECTS_ALL already identifies the subject
    # corresponding to every trial.
    #
    # Removing argsort completely prevents accidental
    # dimensional/indexing problems.
    # ========================================================

    X = X.astype(
        np.float32
    )

    y = y.astype(
        np.int64
    )

    subjects = subjects.astype(
        str
    )

    # ========================================================
    # FINAL REPORT
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL DATASET")
    print("=" * 70)

    print(
        "X shape       :",
        X.shape
    )

    print(
        "y shape       :",
        y.shape
    )

    print(
        "subjects shape:",
        subjects.shape
    )

    print(
        "\nNumber of subjects:",
        len(np.unique(subjects))
    )

    print(
        "Subjects:",
        np.unique(subjects).tolist()
    )

    print("\nSubject distribution:")

    unique_subjects, counts = np.unique(
        subjects,
        return_counts=True
    )

    for sid, count in zip(
        unique_subjects,
        counts
    ):

        print(
            f"  {sid}: {count} trials"
        )

    print("\nClass distribution:")

    for c in range(
        cfg.n_classes
    ):

        print(
            f"  Class {c}: "
            f"{int(np.sum(y == c))}"
        )

    return (
        X,
        y,
        subjects
    )


# ============================================================
# EXECUTE
# ============================================================

X_ALL, y_ALL, SUBJECTS_ALL = build_dataset(
    CFG_DEFAULT
)

print("\n" + "=" * 70)
print("DATASET BUILD SUCCESSFUL")
print("=" * 70)

print(
    "X_ALL        =",
    X_ALL.shape
)

print(
    "y_ALL        =",
    y_ALL.shape
)

print(
    "SUBJECTS_ALL =",
    SUBJECTS_ALL.shape
)

assert len(X_ALL) == len(y_ALL)
assert len(X_ALL) == len(SUBJECTS_ALL)

print(
    "\nSafety check passed:"
)

print(
    "len(X_ALL) == len(y_ALL) == len(SUBJECTS_ALL) =",
    len(X_ALL) == len(y_ALL) == len(SUBJECTS_ALL)
)

Found 18 GDF files.

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A01E.gdf

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A01T.gdf
  X         : (288, 22, 1000)
  y         : (288,)
  subjects  : (288,)
  subject    : A01

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A02E.gdf

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A02T.gdf
  X         : (288, 22, 1000)
  y         : (288,)
  subjects  : (288,)
  subject    : A02

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A03E.gdf

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A03T.gdf
  X         : (288, 22, 1000)
  y         : (288,)
  subjects  : (288,)
  subject    : A03

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A04E.gdf

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A04T.gdf
  X         : (288, 22, 1000)
  y         : (288,)
  subjects  : (288,)
  subject    : A04

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A05E.gdf

Loading:
/Users/ashokvarmabevara/Project2/BCI IV-2a/A05T.

In [17]:

# CELL 7 — PREPROCESSING
def butter_bandpass_filter(X, low, high, fs, order=4):
    sos = signal.butter(order, [low, high], btype="bandpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, X, axis=-1).astype(np.float32)

def preprocess_array(X, cfg: CFG):
    X = np.nan_to_num(X, nan=np.nanmedian(X, axis=-1, keepdims=True))
    if cfg.preprocess_mode == "paper_1_38":
        X = butter_bandpass_filter(X, 1.0, 38.0, cfg.sfreq, order=5)
    elif cfg.preprocess_mode == "paper_8_30":
        X = butter_bandpass_filter(X, 8.0, 30.0, cfg.sfreq, order=4)
    elif cfg.preprocess_mode == "nf_raw":
        # Deliberately no handcrafted filter-bank for the primary NF-EEG branch.
        pass
    else:
        raise ValueError(cfg.preprocess_mode)
    return X.astype(np.float32)

X_ALL = preprocess_array(X_ALL, CFG_DEFAULT)
print("Preprocessed:", X_ALL.shape, float(np.nanstd(X_ALL)))


Preprocessed: (2592, 22, 1000) 4.7515435218811035


In [45]:
# ============================================================
# CELL 8 — SUBJECT SPLITTING / LOSO FOLD SETUP
# ============================================================

def subject_split_indices(subjects, target_subject):

    subjects = np.asarray(subjects).astype(str)

    unique_subjects = np.unique(subjects)

    if target_subject not in unique_subjects:
        raise ValueError(
            f"Target subject {target_subject} not found.\n"
            f"Available subjects: {unique_subjects.tolist()}"
        )

    test_idx = np.flatnonzero(
        subjects == target_subject
    )

    train_idx = np.flatnonzero(
        subjects != target_subject
    )

    if len(test_idx) == 0:
        raise RuntimeError(
            f"No test trials found for {target_subject}"
        )

    if len(train_idx) == 0:
        raise RuntimeError(
            f"No source trials available when holding out {target_subject}"
        )

    return train_idx, test_idx


def inner_source_validation_split(
    subjects,
    train_idx,
    n_val_subjects=2,
    seed=2026
):

    subjects = np.asarray(subjects).astype(str)

    source_subjects = np.unique(
        subjects[train_idx]
    )

    print(
        "Available source subjects:",
        source_subjects.tolist()
    )

    # Need at least one subject left for actual training.
    if len(source_subjects) < n_val_subjects + 1:

        raise ValueError(
            f"Only {len(source_subjects)} source subjects are "
            f"available, but n_val_subjects={n_val_subjects}. "
            f"Need at least {n_val_subjects + 1}."
        )

    rng = np.random.RandomState(seed)

    val_subjects = rng.choice(
        source_subjects,
        size=n_val_subjects,
        replace=False
    )

    val_mask = np.isin(
        subjects[train_idx],
        val_subjects
    )

    val_idx = train_idx[val_mask]
    inner_train_idx = train_idx[~val_mask]

    # Safety checks
    inner_train_subjects = np.unique(
        subjects[inner_train_idx]
    )

    validation_subjects = np.unique(
        subjects[val_idx]
    )

    if len(inner_train_subjects) == 0:
        raise RuntimeError(
            "Inner training set is empty."
        )

    if set(inner_train_subjects) & set(validation_subjects):
        raise RuntimeError(
            "Subject leakage detected between inner train and validation."
        )

    return (
        inner_train_idx,
        val_idx,
        validation_subjects.tolist()
    )


# ============================================================
# IMPORTANT: REFRESH LOSO SUBJECT LIST
# ============================================================

SUBJECTS_ALL = np.asarray(
    SUBJECTS_ALL
).astype(str)

LOSOSUBJECTS = sorted(
    np.unique(SUBJECTS_ALL).tolist()
)

print("\n" + "=" * 70)
print("LOSO SUBJECT SET")
print("=" * 70)

print(
    "Subjects:",
    LOSOSUBJECTS
)

print(
    "Number of subjects:",
    len(LOSOSUBJECTS)
)

print(
    "\nTrials per subject:"
)

for sid, count in zip(
    *np.unique(
        SUBJECTS_ALL,
        return_counts=True
    )
):
    print(
        f"  {sid}: {count}"
    )

assert len(LOSOSUBJECTS) == 9, (
    f"Expected 9 subjects, "
    f"found {len(LOSOSUBJECTS)}"
)

assert len(X_ALL) == len(y_ALL)
assert len(X_ALL) == len(SUBJECTS_ALL)

print(
    "\nLOSO setup is valid."
)


LOSO SUBJECT SET
Subjects: ['A']
Number of subjects: 1

Trials per subject:
  A: 2592


AssertionError: Expected 9 subjects, found 1

In [19]:
# CELL 9 — TRAIN-ONLY NORMALIZATION + ALIGNMENT
@dataclass
class Normalizer:
    mode: str
    mean_: Optional[np.ndarray] = None
    std_: Optional[np.ndarray] = None
    def fit(self, X):
        if self.mode == "train_zscore_global":
            self.mean_ = np.asarray(X.mean(), dtype=np.float32)
            self.std_ = np.asarray(X.std() + 1e-6, dtype=np.float32)
        elif self.mode == "train_zscore_channel":
            self.mean_ = X.mean(axis=(0,2), keepdims=True).astype(np.float32)
            self.std_ = (X.std(axis=(0,2), keepdims=True)+1e-6).astype(np.float32)
        else:
            raise ValueError(self.mode)
        return self
    def transform(self, X):
        return ((X-self.mean_)/self.std_).astype(np.float32)
    def fit_transform(self, X):
        return self.fit(X).transform(X)

@dataclass
class SourceWhitening:
    eps: float = 1e-5
    W_: Optional[np.ndarray] = None
    def fit(self, X):
        C = np.mean([normalized_covariance(t) for t in X], axis=0)
        vals, vecs = np.linalg.eigh(C + self.eps*np.eye(C.shape[0]))
        self.W_ = (vecs @ np.diag(1.0/np.sqrt(vals+self.eps)) @ vecs.T).astype(np.float32)
        return self
    def transform(self, X):
        return np.einsum('ij,njt->nit', self.W_, X).astype(np.float32)

def euclidean_alignment_target_calibration(X):
    C = np.mean([normalized_covariance(t) for t in X], axis=0)
    vals, vecs = np.linalg.eigh(C + 1e-5*np.eye(C.shape[0]))
    W = (vecs @ np.diag(1.0/np.sqrt(vals+1e-5)) @ vecs.T).astype(np.float32)
    return np.einsum('ij,njt->nit', W, X).astype(np.float32), W

def subject_batch_mmd(features, subject_ids):
    ids = np.asarray(subject_ids)
    uniq = np.unique(ids)
    if len(uniq) < 2: return features.new_tensor(0.0)
    vals=[]
    for i in range(len(uniq)):
        for j in range(i+1,len(uniq)):
            a=features[torch.as_tensor(ids==uniq[i], device=features.device)]
            b=features[torch.as_tensor(ids==uniq[j], device=features.device)]
            if len(a)>1 and len(b)>1: vals.append(mmd_rbf(a,b))
    return torch.stack(vals).mean() if vals else features.new_tensor(0.0)


In [20]:

# CELL 10 — NF-EEG PATCH / TOKENIZER
class PatchEmbedding2D(nn.Module):
    def __init__(self, in_ch=1, dim=128,
                 spatial_patch=4, temporal_patch=25,
                 spatial_stride=4, temporal_stride=25):
        super().__init__()
        self.proj = nn.Conv2d(
            in_ch, dim,
            kernel_size=(spatial_patch, temporal_patch),
            stride=(spatial_stride, temporal_stride)
        )

    def forward(self, x):
        # x: [B,1,22,1000]
        z = self.proj(x)
        b, d, h, w = z.shape
        tokens = z.flatten(2).transpose(1, 2)
        return tokens, (h, w)

class PositionalEncodingLearned(nn.Module):
    def __init__(self, max_tokens=200, dim=128):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_tokens, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        return x + self.pos[:, :x.size(1)]


In [21]:

# CELL 11 — EEGNET BRANCH
class EEGNetBranch(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        F1, D, F2, K = cfg.eegnet_f1, cfg.eegnet_d, cfg.eegnet_f2, cfg.eegnet_kernel
        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, K), padding=(0, K // 2), bias=False),
            nn.BatchNorm2d(F1)
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(cfg.n_channels, 1),
                      groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(cfg.dropout)
        )
        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 15),
                      padding=(0, 7), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(cfg.dropout)
        )
        self.proj = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(F2, cfg.fusion_dim)
        )

    def forward(self, x):
        z = self.temporal(x)
        z = self.spatial(z)
        z = self.separable(z)
        return self.proj(z)


In [22]:

# CELL 12 — MULTI-SCALE NF-EEG CNN
def make_activation(name):
    name = name.lower()
    if name == "relu": return nn.ReLU(inplace=True)
    if name == "elu": return nn.ELU(inplace=True)
    if name == "gelu": return nn.GELU()
    if name == "tanh": return nn.Tanh()
    raise ValueError(name)

class ChannelGate(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.net(x)

class MultiScaleBranch(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        C = cfg.multiscale_channels
        self.branches = nn.ModuleList()
        for k in cfg.multiscale_kernels:
            self.branches.append(nn.Sequential(
                nn.Conv2d(1, C, kernel_size=(1, k), padding=(0, k//2), bias=False),
                nn.BatchNorm2d(C),
                make_activation(cfg.activation),
                nn.Conv2d(C, C, kernel_size=(cfg.n_channels, 1), groups=C, bias=False),
                nn.BatchNorm2d(C),
                make_activation(cfg.activation),
                nn.Conv2d(C, C, kernel_size=1, bias=False),
                nn.BatchNorm2d(C),
                make_activation(cfg.activation),
            ))
        self.merge = nn.Sequential(
            nn.Conv2d(C * len(cfg.multiscale_kernels), C * 2, kernel_size=1, bias=False),
            nn.BatchNorm2d(C * 2),
            make_activation(cfg.activation),
            nn.Conv2d(C * 2, C * 2, kernel_size=(1, 7), padding=(0, 3), groups=C * 2, bias=False),
            nn.BatchNorm2d(C * 2),
            make_activation(cfg.activation),
        )
        self.attn = ChannelGate(C * 2) if cfg.use_channel_attention and cfg.attention_type == "gate" else nn.Identity()
        self.pool = nn.AvgPool2d(kernel_size=(1, 4), stride=(1, 4))
        self.dropout = nn.Dropout2d(cfg.dropout)
        self.project = nn.Linear(C * 2, cfg.fusion_dim)

    def forward(self, x):
        outs = [b(x) for b in self.branches]
        z = torch.cat(outs, dim=1)
        z = self.merge(z)
        z = self.attn(z)
        z = self.pool(z)
        z = self.dropout(z)
        pooled = z.mean(dim=(2, 3))
        vec = self.project(pooled)
        return z, vec


In [23]:

# CELL 13 — ATTENTION BLOCKS
class SEBlock(nn.Module):
    def __init__(self, dim, reduction=8):
        super().__init__()
        hidden = max(dim // reduction, 8)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.net(x.mean(dim=1))

class CBAMTokenGate(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(dim // 2, dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.mlp(x.mean(dim=1))

def build_feature_attention(cfg: CFG, dim: int):
    if not cfg.use_channel_attention or cfg.attention_type == "none":
        return nn.Identity()
    if cfg.attention_type == "se":
        return SEBlock(dim)
    if cfg.attention_type == "cbam":
        return CBAMTokenGate(dim)
    return nn.Identity()


In [24]:

# CELL 14 — TRANSFORMER RELATIONAL ENCODER
class TransformerRelational(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.pos = PositionalEncodingLearned(cfg.max_tokens, cfg.transformer_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=cfg.transformer_dim,
            nhead=cfg.transformer_heads,
            dim_feedforward=cfg.transformer_ff,
            dropout=cfg.transformer_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.transformer_layers)
        self.norm = nn.LayerNorm(cfg.transformer_dim)
        self.proj = nn.Linear(cfg.transformer_dim, cfg.fusion_dim)

    def forward(self, tokens):
        if tokens.size(1) > CFG_DEFAULT.max_tokens:
            idx = torch.linspace(0, tokens.size(1)-1, CFG_DEFAULT.max_tokens, device=tokens.device).long()
            tokens = tokens[:, idx]
        z = self.pos(tokens)
        z = self.encoder(z)
        z = self.norm(z)
        pooled = z.mean(dim=1)
        return z, self.proj(pooled)


In [25]:

# CELL 15 — BiLSTM TEMPORAL MODEL
class BiLSTMHead(nn.Module):
    def __init__(self, dim, cfg: CFG):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=dim,
            hidden_size=cfg.bilstm_hidden,
            num_layers=cfg.bilstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.bilstm_dropout if cfg.bilstm_layers > 1 else 0.0
        )
        self.proj = nn.Linear(cfg.bilstm_hidden * 2, cfg.fusion_dim)

    def forward(self, tokens):
        z, _ = self.lstm(tokens)
        pooled = z.mean(dim=1)
        return z, self.proj(pooled)


In [26]:

# CELL 16 — GATED FUSION
class GatedFusion(nn.Module):
    def __init__(self, n_branches: int, dim: int):
        super().__init__()
        self.n = n_branches
        self.gate = nn.Sequential(
            nn.Linear(dim * n_branches, dim),
            nn.GELU(),
            nn.Linear(dim, n_branches)
        )
        self.out = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU()
        )

    def forward(self, feats: List[torch.Tensor]):
        if len(feats) == 1:
            return self.out(feats[0]), torch.ones(feats[0].size(0), 1, device=feats[0].device)
        x = torch.cat(feats, dim=-1)
        weights = torch.softmax(self.gate(x), dim=-1)
        stacked = torch.stack(feats, dim=1)
        fused = (stacked * weights.unsqueeze(-1)).sum(dim=1)
        return self.out(fused), weights


In [27]:

# CELL 17 — DISCRIMINATIVE LOSSES
class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim, alpha=0.5):
        super().__init__()
        self.num_classes = num_classes
        self.alpha = alpha
        self.register_buffer("centers", torch.zeros(num_classes, feat_dim))

    def forward(self, features, labels):
        centers_batch = self.centers[labels]
        loss = ((features - centers_batch) ** 2).sum(dim=1).mean()
        with torch.no_grad():
            for c in labels.unique():
                mask = labels == c
                delta = features[mask].mean(dim=0) - self.centers[c]
                self.centers[c] += self.alpha * delta
        return loss

def supervised_contrastive_loss(features, labels, temperature=0.1):
    z = F.normalize(features, dim=-1)
    sim = torch.matmul(z, z.T) / temperature
    mask = labels[:, None].eq(labels[None, :]).float()
    logits_mask = torch.ones_like(mask) - torch.eye(len(labels), device=labels.device)
    mask = mask * logits_mask
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    exp_sim = torch.exp(sim) * logits_mask
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return (-(mask * log_prob).sum(dim=1) / denom).mean()

def mmd_rbf(x, y, sigma=1.0):
    def k(a, b):
        aa = (a*a).sum(1, keepdim=True)
        bb = (b*b).sum(1, keepdim=True).T
        dist = aa + bb - 2*a@b.T
        return torch.exp(-dist / (2*sigma*sigma))
    return k(x, x).mean() + k(y, y).mean() - 2*k(x, y).mean()


In [28]:
# CELL 18 — PAPER-STYLE CLASS-CONDITIONED FBGAN
class FBGANGenerator(nn.Module):
    def __init__(self, latent_dim=1600):
        super().__init__()
        self.fc=nn.Linear(latent_dim,128*16*125)
        self.net=nn.Sequential(
            nn.ConvTranspose2d(128,128,(3,15),(1,3),(1,6)), nn.BatchNorm2d(128), nn.LeakyReLU(.2,True),
            nn.ConvTranspose2d(128,128,(3,15),(1,3),(1,6)), nn.BatchNorm2d(128), nn.LeakyReLU(.2,True),
            nn.ConvTranspose2d(128,64,(3,5),(1,2),(1,2)), nn.BatchNorm2d(64), nn.LeakyReLU(.2,True),
            nn.ConvTranspose2d(64,32,(4,5),(2,1),(1,2)), nn.BatchNorm2d(32), nn.LeakyReLU(.2,True),
            nn.ConvTranspose2d(32,1,(1,2),(1,1),(0,0)) )
    def forward(self,z):
        x=self.fc(z).view(z.size(0),128,16,125)
        return F.interpolate(self.net(x),size=(22,1000),mode='bilinear',align_corners=False).squeeze(1)

class EEGDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,10,(1,23),padding=(0,11)), nn.LeakyReLU(.2,True),
            nn.Conv2d(10,30,(22,1)), nn.LeakyReLU(.2,True),
            nn.Conv2d(30,30,(1,17),padding=(0,8)), nn.LeakyReLU(.2,True),
            nn.AdaptiveMaxPool2d((1,1)), nn.Flatten(), nn.Linear(30,1))
    def forward(self,x): return self.net(x.unsqueeze(1)).squeeze(-1)

class SparseFeatureDiscriminator(nn.Module):
    def __init__(self,feature_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(feature_dim,128),nn.LeakyReLU(.2,True),nn.Linear(128,64),nn.LeakyReLU(.2,True),nn.Linear(64,1))
    def forward(self,x): return self.net(x).squeeze(-1)

class FBGANClassTrainer:
    def __init__(self,sparse_transform_fn,latent_dim=1600,lr=1e-4,device=DEVICE):
        self.G=FBGANGenerator(latent_dim).to(device); self.D_phi=EEGDiscriminator().to(device); self.D_psi=None
        self.device=device; self.latent_dim=latent_dim; self.lr=lr; self.sparse_transform_fn=sparse_transform_fn
    def fit(self,X_real,sparse_dim,epochs=100,batch_size=5):
        self.D_psi=SparseFeatureDiscriminator(sparse_dim).to(self.device)
        optG=torch.optim.Adam(self.G.parameters(),lr=self.lr,betas=(.5,.999)); opt1=torch.optim.Adam(self.D_phi.parameters(),lr=self.lr,betas=(.5,.999)); opt2=torch.optim.Adam(self.D_psi.parameters(),lr=self.lr,betas=(.5,.999))
        dl=DataLoader(torch.utils.data.TensorDataset(torch.tensor(X_real,dtype=torch.float32)),batch_size=batch_size,shuffle=True)
        bce=nn.BCEWithLogitsLoss(); hist=[]
        for ep in range(epochs):
            for (real,) in dl:
                real=real.to(self.device); z=torch.randn(len(real),self.latent_dim,device=self.device); fake=self.G(z)
                opt1.zero_grad(set_to_none=True); lr=self.D_phi(real); lf=self.D_phi(fake.detach()); ld1=bce(lr,torch.ones_like(lr))+bce(lf,torch.zeros_like(lf)); ld1.backward(); opt1.step()
                rs=torch.tensor(self.sparse_transform_fn(real.detach().cpu().numpy()),dtype=torch.float32,device=self.device); fs=torch.tensor(self.sparse_transform_fn(fake.detach().cpu().numpy()),dtype=torch.float32,device=self.device)
                opt2.zero_grad(set_to_none=True); dr=self.D_psi(rs); df=self.D_psi(fs.detach()); ld2=bce(dr,torch.ones_like(dr))+bce(df,torch.zeros_like(df)); ld2.backward(); opt2.step()
                optG.zero_grad(set_to_none=True); fake=self.G(z); lg=bce(self.D_phi(fake),torch.ones(len(real),device=self.device)); fs=torch.tensor(self.sparse_transform_fn(fake.detach().cpu().numpy()),dtype=torch.float32,device=self.device); lg=lg+bce(self.D_psi(fs),torch.ones(len(real),device=self.device)); lg.backward(); optG.step()
            hist.append((ep,float(lg.detach().cpu()),float(ld1.detach().cpu()),float(ld2.detach().cpu())))
        return pd.DataFrame(hist,columns=['epoch','G','D_phi','D_psi'])
    def sample(self,n):
        self.G.eval(); out=[]
        with torch.no_grad():
            for i in range(0,n,32):
                z=torch.randn(min(32,n-i),self.latent_dim,device=self.device); out.append(self.G(z).cpu().numpy())
        return np.concatenate(out,axis=0)

def train_fbgan_per_class(X_cal,y_cal,cfg):
    Xfake=[]; yfake=[]; histories=[]
    for c in range(cfg.n_classes):
        Xc=X_cal[y_cal==c]
        feat,filters,_=sparse_csp_features(Xc,np.full(len(Xc),c),bands=FB_BANDS,n_filters=4,fit_lasso=False)
        fn=make_sparse_transform(filters)
        tr=FBGANClassTrainer(fn,cfg.fbgan_latent_dim,cfg.fbgan_lr,DEVICE)
        hist=tr.fit(Xc,feat.shape[1],epochs=5 if cfg.fast_dev_run else cfg.fbgan_epochs,batch_size=cfg.fbgan_batch_size)
        fx=tr.sample(cfg.fbgan_samples_per_class)
        Xfake.append(fx); yfake.append(np.full(len(fx),c,dtype=np.int64)); histories.append(hist)
        del tr; gc.collect()
        if DEVICE.type=='mps': torch.mps.empty_cache()
    return np.concatenate(Xfake),np.concatenate(yfake),histories


In [29]:

# CELL 19 — CSP/LASSO + SYNTHETIC EEG QUALITY CONTROL
FB_BANDS = [(1,4),(4,8),(8,12),(12,16),(16,20),(20,24),(24,28),(28,32),(32,35),(35,38)]

def bandpass_trials(X, lo, hi, fs=250):
    sos = signal.butter(4, [lo, hi], btype="bandpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, X, axis=-1).astype(np.float32)

def normalized_covariance(trial):
    c = np.cov(trial)
    return c / (np.trace(c) + 1e-8)

def ovr_csp_filters(X, y, n_filters=4):
    # One-vs-rest generalized eigenvectors per class, based only on supplied training/calibration data.
    filters = []
    for c in range(CFG_DEFAULT.n_classes):
        Xc = X[y == c]
        Xr = X[y != c]
        Rc = np.mean([normalized_covariance(t) for t in Xc], axis=0)
        Rr = np.mean([normalized_covariance(t) for t in Xr], axis=0)
        vals, vecs = linalg.eigh(Rc, Rc + Rr + 1e-6*np.eye(Rc.shape[0]))
        idx = np.argsort(vals)
        idx = np.r_[idx[:n_filters], idx[-n_filters:]]
        filters.append(vecs[:, idx].T)
    return np.concatenate(filters, axis=0)

def sparse_csp_features(X, y, bands=FB_BANDS, n_filters=4, fit_lasso=True):
    feats, bank_filters = [], []
    for lo, hi in bands:
        Xb = bandpass_trials(X, lo, hi, CFG_DEFAULT.sfreq)
        W = ovr_csp_filters(Xb, y, n_filters)
        Z = np.asarray([
            np.var(W @ t, axis=1) for t in Xb
        ])
        Z = np.log(Z + 1e-8)
        feats.append(Z)
        bank_filters.append(W)
    Fbank = np.concatenate(feats, axis=1)
    if fit_lasso:
        scaler = StandardScaler()
        Fs = scaler.fit_transform(Fbank)
        clf = LogisticRegression(
            penalty="l1", solver="liblinear", C=0.2,
            multi_class="ovr", max_iter=2000, random_state=CFG_DEFAULT.seed
        )
        clf.fit(Fs, y)
        support = np.any(np.abs(clf.coef_) > 1e-7, axis=0)
    else:
        scaler, clf, support = None, None, np.ones(Fbank.shape[1], dtype=bool)
    return Fbank[:, support], bank_filters, (scaler, clf, support)

def make_sparse_transform(bank_filters):
    Ws = np.concatenate(bank_filters, axis=0)
    def fn(X):
        out = []
        for trial in X:
            vals = []
            for W in bank_filters:
                vals.append(np.var(W @ trial, axis=1))
            out.append(np.concatenate(vals))
        return np.log(np.asarray(out) + 1e-8)
    return fn

def quality_scores(real, fake, eps=1e-8):
    # Independent QC: time statistics, PSD, covariance, diversity.
    real = np.asarray(real); fake = np.asarray(fake)
    mean_dist = abs(real.mean() - fake.mean()) / (real.std() + eps)
    std_ratio = fake.std() / (real.std() + eps)
    f, p_real = signal.welch(real.mean(axis=1), fs=CFG_DEFAULT.sfreq, nperseg=256, axis=-1)
    _, p_fake = signal.welch(fake.mean(axis=1), fs=CFG_DEFAULT.sfreq, nperseg=256, axis=-1)
    psd_dist = np.mean(np.abs(np.log(p_real + eps) - np.log(p_fake + eps)))
    c_real = np.mean([normalized_covariance(t) for t in real], axis=0)
    c_fake = np.mean([normalized_covariance(t) for t in fake], axis=0)
    cov_dist = np.linalg.norm(c_real - c_fake) / (np.linalg.norm(c_real) + eps)
    diversity = float(np.mean(np.std(fake, axis=0)))
    return {
        "mean_dist": float(mean_dist),
        "std_ratio": float(std_ratio),
        "psd_log_distance": float(psd_dist),
        "cov_relative_distance": float(cov_dist),
        "diversity": diversity
    }


In [30]:

# CELL 20 — HYBRID MODEL
class HybridNFEEGModel(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.cfg = cfg
        self.patch = PatchEmbedding2D(
            1, cfg.token_dim,
            cfg.patch_spatial, cfg.patch_temporal,
            cfg.patch_stride_spatial, cfg.patch_stride_temporal
        ) if cfg.use_nf_eeg else None
        self.raw_proj = nn.Linear(cfg.token_dim, cfg.fusion_dim) if cfg.use_nf_eeg else None

        self.ms = MultiScaleBranch(cfg) if cfg.use_multiscale else None
        self.eegnet = EEGNetBranch(cfg) if cfg.use_eegnet else None
        self.transformer = TransformerRelational(cfg) if cfg.use_transformer else None
        self.bilstm = BiLSTMHead(cfg.transformer_dim, cfg) if (cfg.use_bilstm and cfg.use_transformer) else None

        self.rel_gate = build_feature_attention(cfg, cfg.transformer_dim)
        self.ms_to_tokens = nn.Conv2d(cfg.multiscale_channels*2, cfg.transformer_dim, 1) if self.ms and self.transformer else None

        branch_count = 0
        if cfg.use_nf_eeg: branch_count += 1
        if cfg.use_eegnet: branch_count += 1
        if cfg.use_multiscale: branch_count += 1
        if cfg.use_transformer: branch_count += 1
        if cfg.use_bilstm: branch_count += 1
        branch_count = max(branch_count, 1)

        self.fusion = GatedFusion(branch_count, cfg.fusion_dim)
        self.head = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim),
            nn.LayerNorm(cfg.fusion_dim),
            make_activation(cfg.activation),
            nn.Dropout(0.5),
            nn.Linear(cfg.fusion_dim, cfg.n_classes)
        )

    def forward(self, x, return_aux=False):
        feats = []
        aux = {}

        if self.patch is not None:
            raw_tokens, _ = self.patch(x)
            raw_vec = self.raw_proj(raw_tokens.mean(dim=1))
            feats.append(raw_vec)
            aux["raw_tokens"] = raw_tokens

        ms_tokens = None
        if self.ms is not None:
            ms_map, ms_vec = self.ms(x)
            feats.append(ms_vec)
            aux["ms_map"] = ms_map
            if self.transformer is not None:
                ms_tokens = self.ms_to_tokens(ms_map).flatten(2).transpose(1, 2)

        if self.eegnet is not None:
            eeg_vec = self.eegnet(x)
            feats.append(eeg_vec)
            aux["eegnet_vec"] = eeg_vec

        rel_tokens = None
        rel_vec = None
        if self.transformer is not None:
            if ms_tokens is None:
                tokens, _ = self.patch(x)
                tokens = tokens[:, :self.cfg.max_tokens]
            else:
                tokens = ms_tokens[:, :self.cfg.max_tokens]
                tokens = tokens[..., :self.cfg.transformer_dim]
            rel_tokens, rel_vec = self.transformer(tokens)
            feats.append(rel_vec)
            aux["rel_tokens"] = rel_tokens

            if self.bilstm is not None:
                rnn_tokens, rnn_vec = self.bilstm(rel_tokens)
                feats.append(rnn_vec)
                aux["bilstm_tokens"] = rnn_tokens

        fused, weights = self.fusion(feats)
        logits = self.head(fused)
        aux["fusion_weights"] = weights
        aux["embedding"] = fused
        return (logits, aux) if return_aux else logits


In [31]:

# CELL 21 — TRAINING UTILITIES
class EEGTensorDataset(Dataset):
    def __init__(self, X, y, subjects=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.subjects = np.asarray(subjects) if subjects is not None else None
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = (self.X[i], self.y[i])
        if self.subjects is not None:
            item += (self.subjects[i],)
        return item

def make_loader(X, y, batch_size, shuffle=True, subjects=None, cfg=CFG_DEFAULT):
    ds = EEGTensorDataset(X, y, subjects)
    sampler = None
    if shuffle:
        counts = np.bincount(y, minlength=cfg.n_classes).astype(np.float64)
        w = 1.0 / np.maximum(counts, 1)
        sample_w = w[y]
        sampler = WeightedRandomSampler(
            torch.as_tensor(sample_w, dtype=torch.double),
            num_samples=len(y), replacement=True
        )
    return DataLoader(
        ds, batch_size=batch_size, shuffle=(sampler is None and shuffle),
        sampler=sampler, num_workers=cfg.num_workers,
        pin_memory=False, drop_last=False
    )

def warmup_cosine_lambda(epoch, total_epochs, warmup_epochs):
    if epoch < warmup_epochs:
        return max((epoch + 1) / max(warmup_epochs,1), 1e-4)
    progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
    return 0.5 * (1 + math.cos(math.pi * progress))

def model_parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def batch_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }

def confidence_interval_95(values):
    values = np.asarray(values, dtype=float)
    mean = values.mean()
    if len(values) < 2:
        return mean, np.nan, np.nan
    sem = stats.sem(values)
    tcrit = stats.t.ppf(0.975, len(values)-1)
    return mean, mean-tcrit*sem, mean+tcrit*sem


In [32]:
# CELL 22 — VALIDATION / TRAINING LOOP

def evaluate_model(model,loader,device=DEVICE):
    model.eval(); ys=[]; ps=[]; embs=[]; loss_sum=0.; n=0; ce=nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            xb,yb=batch[:2]; xb=xb.to(device).unsqueeze(1); yb=yb.to(device); logits,aux=model(xb,return_aux=True); loss=ce(logits,yb); bs=len(yb); loss_sum+=float(loss.cpu())*bs; n+=bs; ys+=yb.cpu().numpy().tolist(); ps+=logits.argmax(1).cpu().numpy().tolist(); embs.append(aux['embedding'].cpu().numpy())
    m=batch_metrics(np.asarray(ys),np.asarray(ps)); m['loss']=loss_sum/max(n,1); return m,np.asarray(ys),np.asarray(ps),np.concatenate(embs)

def train_one_fold(model,train_loader,val_loader,cfg,center_loss=None):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay); sched=torch.optim.lr_scheduler.LambdaLR(opt,lambda e:warmup_cosine_lambda(e,cfg.epochs,cfg.warmup_epochs)); ce=nn.CrossEntropyLoss(); best={'score':-np.inf,'state':None,'epoch':-1,'metrics':None}; hist=[]; no_imp=0
    for ep in range(cfg.epochs):
        model.train(); yt=[]; yp=[]; total=0.; seen=0; opt.zero_grad(set_to_none=True)
        for step,batch in enumerate(train_loader):
            xb,yb=batch[:2]; sid=batch[2] if len(batch)>2 else None; xb=xb.to(DEVICE).unsqueeze(1); yb=yb.to(DEVICE); logits,aux=model(xb,return_aux=True); loss=ce(logits,yb)
            if cfg.use_center_loss and center_loss is not None: loss=loss+cfg.center_lambda*center_loss(aux['embedding'],yb)
            if cfg.use_supcon: loss=loss+cfg.supcon_lambda*supervised_contrastive_loss(aux['embedding'],yb)
            if cfg.use_domain_alignment and sid is not None: loss=loss+cfg.domain_lambda*subject_batch_mmd(aux['embedding'],np.asarray(sid))
            (loss/cfg.grad_accum_steps).backward()
            if (step+1)%cfg.grad_accum_steps==0: nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); opt.step(); opt.zero_grad(set_to_none=True)
            bs=len(yb); total+=float(loss.detach().cpu())*bs; seen+=bs; yt+=yb.detach().cpu().numpy().tolist(); yp+=logits.argmax(1).detach().cpu().numpy().tolist()
        sched.step(); tm=batch_metrics(np.asarray(yt),np.asarray(yp)); tm['loss']=total/max(seen,1); vm,_,_,_=evaluate_model(model,val_loader); row={'epoch':ep,'train_loss':tm['loss'],'train_accuracy':tm['accuracy'],'val_loss':vm['loss'],'val_accuracy':vm['accuracy'],'val_balanced_accuracy':vm['balanced_accuracy'],'lr':opt.param_groups[0]['lr']}; hist.append(row)
        score=vm['balanced_accuracy']
        if score>best['score']: best={'score':score,'state':copy.deepcopy(model.state_dict()),'epoch':ep,'metrics':vm}; no_imp=0
        else: no_imp+=1
        if no_imp>=cfg.patience or (cfg.fast_dev_run and ep>=2): break
    model.load_state_dict(best['state']); return model,pd.DataFrame(hist),best


In [33]:
# CELL 23 — STRICT LOSO

def run_strict_loso(cfg,X,y,subjects):
    results=[]; folds={}
    for target in LOSOSUBJECTS:
        train_idx,test_idx=subject_split_indices(subjects,target); inner_train,val_idx,val_subjects=inner_source_validation_split(subjects,train_idx,cfg.validation_subjects_per_fold,cfg.inner_seed)
        norm=Normalizer(cfg.normalize); Xtr=norm.fit_transform(X[inner_train]); Xva=norm.transform(X[val_idx]); Xte=norm.transform(X[test_idx])
        aligner=None
        if cfg.strict_source_whitening: aligner=SourceWhitening().fit(Xtr); Xtr=aligner.transform(Xtr); Xva=aligner.transform(Xva); Xte=aligner.transform(Xte)
        model=HybridNFEEGModel(cfg); center=CenterLoss(cfg.n_classes,cfg.fusion_dim,cfg.center_lr).to(DEVICE) if cfg.use_center_loss else None
        tr_loader=make_loader(Xtr,y[inner_train],cfg.batch_size,True,subjects=subjects[inner_train] if cfg.use_domain_alignment else None,cfg=cfg); va_loader=make_loader(Xva,y[val_idx],cfg.batch_size,False,cfg=cfg); te_loader=make_loader(Xte,y[test_idx],cfg.batch_size,False,cfg=cfg)
        t0=time.time(); model,hist,best=train_one_fold(model,tr_loader,va_loader,cfg,center); train_s=time.time()-t0; met,yt,yp,emb=evaluate_model(model,te_loader); cm=confusion_matrix(yt,yp,labels=list(range(cfg.n_classes)))
        results.append({'subject':target,**met,'val_best_balanced_accuracy':best['metrics']['balanced_accuracy'],'best_epoch':best['epoch'],'params_m':model_parameter_count(model)/1e6,'train_seconds':train_s,'val_subjects':','.join(val_subjects)})
        ck=Path(cfg.checkpoint_dir)/f'strict_loso_{target}.pt'; torch.save({'model_state':model.state_dict(),'config':asdict(cfg),'normalizer_mean':norm.mean_,'normalizer_std':norm.std_,'source_whitening':None if aligner is None else aligner.W_,'subject':target},ck)
        folds[target]={'history':hist,'metrics':met,'confusion_matrix':cm,'y_true':yt,'y_pred':yp,'embeddings':emb,'normalizer':norm,'aligner':aligner,'checkpoint':str(ck)}
        del model,tr_loader,va_loader,te_loader; gc.collect();
        if DEVICE.type=='mps': torch.mps.empty_cache()
    df=pd.DataFrame(results); mean,lo,hi=confidence_interval_95(df.accuracy.values); summ={'mean_accuracy':mean,'std_accuracy':float(df.accuracy.std(ddof=1)),'ci95_low':lo,'ci95_high':hi,'median_accuracy':float(df.accuracy.median()),'min_subject_accuracy':float(df.accuracy.min()),'max_subject_accuracy':float(df.accuracy.max())}; return df,summ,folds


In [34]:
# CELL 24 — TARGET-ADAPTATION LOSO

def make_target_calibration_split(test_idx,y,per_class=12,seed=2026):
    rng=np.random.RandomState(seed); calib=[]; test_keep=[]
    for c in range(CFG_DEFAULT.n_classes):
        ids=test_idx[y[test_idx]==c].copy(); rng.shuffle(ids); take=min(per_class,len(ids)); calib+=ids[:take].tolist(); test_keep+=ids[take:].tolist()
    return np.asarray(calib),np.asarray(test_keep)

def conservative_finetune(model,X_adapt,y_adapt,X_val,y_val,cfg,stage_epochs=(15,25,35),stage_lrs=(5e-5,3e-5,1e-5)):
    best_state=copy.deepcopy(model.state_dict()); best=-np.inf; all_hist=[]
    for stage,(epochs,lr) in enumerate(zip(stage_epochs,stage_lrs),1):
        freeze_for_stage(model,stage); params=[p for p in model.parameters() if p.requires_grad]; opt=torch.optim.AdamW(params,lr=lr,weight_decay=cfg.weight_decay); tr=make_loader(X_adapt,y_adapt,cfg.batch_size,True,cfg=cfg); va=make_loader(X_val,y_val,cfg.batch_size,False,cfg=cfg)
        for ep in range(epochs):
            model.train()
            for xb,yb in tr:
                xb=xb.to(DEVICE).unsqueeze(1); yb=yb.to(DEVICE); opt.zero_grad(set_to_none=True); loss=F.cross_entropy(model(xb),yb); loss.backward(); nn.utils.clip_grad_norm_(params,cfg.grad_clip); opt.step()
            vm,*_=evaluate_model(model,va); all_hist.append({'stage':stage,'epoch':ep,**vm,'lr':lr})
            if vm['balanced_accuracy']>best: best=vm['balanced_accuracy']; best_state=copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); return model,pd.DataFrame(all_hist)

def run_target_adaptation_loso(cfg,X,y,subjects):
    if not cfg.use_target_adaptation: raise ValueError('Set use_target_adaptation=True')
    rows=[]
    for target in LOSOSUBJECTS:
        source_idx,target_all=subject_split_indices(subjects,target); calib_idx,test_idx=make_target_calibration_split(target_all,y,cfg.target_calibration_trials_per_class,cfg.seed); inner_train,val_idx,val_subjects=inner_source_validation_split(subjects,source_idx,cfg.validation_subjects_per_fold,cfg.inner_seed)
        norm=Normalizer(cfg.normalize).fit(X[inner_train]); Xtr=norm.transform(X[inner_train]); Xva=norm.transform(X[val_idx]); Xcal=norm.transform(X[calib_idx]); Xte=norm.transform(X[test_idx])
        if cfg.target_use_ea: Xcal,W=euclidean_alignment_target_calibration(Xcal); Xte=np.einsum('ij,njt->nit',W,Xte).astype(np.float32)
        model=HybridNFEEGModel(cfg); tr=make_loader(Xtr,y[inner_train],cfg.batch_size,True,subjects=subjects[inner_train] if cfg.use_domain_alignment else None,cfg=cfg); va=make_loader(Xva,y[val_idx],cfg.batch_size,False,cfg=cfg); model,pre_hist,pre_best=train_one_fold(model,tr,va,cfg)
        Xfake,yfake,gan_hist=train_fbgan_per_class(Xcal,y[calib_idx],cfg)
        qc=[]
        for c in range(cfg.n_classes):
            rc=Xcal[y[calib_idx]==c]; fc=Xfake[yfake==c]; q=quality_scores(rc,fc[:len(rc)]); q['class']=c; qc.append(q)
        qc_df=pd.DataFrame(qc)
        n_keep=min(len(Xfake),int(len(Xtr)*cfg.fbgan_aug_ratio)); ids=[]
        for c in range(cfg.n_classes):
            ic=np.where(yfake==c)[0]; ids+=ic[:min(len(ic),max(1,n_keep//cfg.n_classes))].tolist()
        ids=np.asarray(ids); Xadapt=np.concatenate([Xtr,Xfake[ids]],axis=0); yadapt=np.concatenate([y[inner_train],yfake[ids]],axis=0)
        model,_=conservative_finetune(model,Xadapt,yadapt,Xva,y[val_idx],cfg); te=make_loader(Xte,y[test_idx],cfg.batch_size,False,cfg=cfg); met,yt,yp,_=evaluate_model(model,te)
        rows.append({'subject':target,**met,'calibration_trials':len(calib_idx),'synthetic_trials_used':len(ids),'qc_mean_cov_distance':float(qc_df.cov_relative_distance.mean()),'qc_mean_psd_distance':float(qc_df.psd_log_distance.mean())})
        del model,tr,va,te; gc.collect();
        if DEVICE.type=='mps': torch.mps.empty_cache()
    return pd.DataFrame(rows)


In [35]:

# CELL 25 — CONSERVATIVE FINE-TUNING
def freeze_for_stage(model, stage: int):
    for p in model.parameters():
        p.requires_grad = False

    # Stage 1: heads/fusion/projectors
    for name, p in model.named_parameters():
        if any(k in name for k in ["fusion", "head", "proj"]):
            p.requires_grad = True

    if stage >= 2:
        for name, p in model.named_parameters():
            if any(k in name for k in ["ms.", "eegnet.", "patch."]):
                p.requires_grad = True

    if stage >= 3:
        for name, p in model.named_parameters():
            if "transformer" in name:
                p.requires_grad = True

def conservative_finetune(model, X_adapt, y_adapt, X_val, y_val, cfg: CFG,
                          stage_epochs=(15, 25, 35), stage_lrs=(5e-5, 3e-5, 1e-5)):
    history_all = []
    best_state = copy.deepcopy(model.state_dict())
    best_score = -np.inf

    for stage, (epochs, lr) in enumerate(zip(stage_epochs, stage_lrs), start=1):
        freeze_for_stage(model, stage)
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=cfg.weight_decay)
        tr_loader = make_loader(X_adapt, y_adapt, cfg.batch_size, True, cfg=cfg)
        va_loader = make_loader(X_val, y_val, cfg.batch_size, False, cfg=cfg)

        for ep in range(epochs):
            model.train()
            for xb, yb in tr_loader:
                xb, yb = xb.to(DEVICE).unsqueeze(1), yb.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(params, cfg.grad_clip)
                opt.step()

            vm, *_ = evaluate_model(model, va_loader)
            history_all.append({"stage": stage, "epoch": ep, **vm, "lr": lr})
            if vm["balanced_accuracy"] > best_score:
                best_score = vm["balanced_accuracy"]
                best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history_all)


In [36]:

# CELL 26 — ENSEMBLE FUSION
class ProbabilityEnsemble:
    def __init__(self, weights=None):
        self.weights = weights

    def fit(self, prob_list, y):
        n = len(prob_list)
        if n == 1:
            self.weights = np.ones(1)
            return self
        # Simple validation-only projected search.
        best_acc, best_w = -np.inf, None
        grid = np.linspace(0.0, 1.0, 11)
        for w0 in grid:
            if n == 2:
                w = np.array([w0, 1-w0])
                p = sum(w[i]*prob_list[i] for i in range(n))
                acc = accuracy_score(y, p.argmax(1))
                if acc > best_acc:
                    best_acc, best_w = acc, w
        if best_w is None:
            self.weights = np.ones(n)/n
        else:
            self.weights = best_w
        return self

    def predict_proba(self, prob_list):
        w = self.weights if self.weights is not None else np.ones(len(prob_list))/len(prob_list)
        return sum(w[i]*prob_list[i] for i in range(len(prob_list)))

# IMPORTANT: fit ensemble weights only on inner validation subjects.


In [37]:
# CELL 27 — ABLATION + CLASSICAL BASELINES
ABLATIONS={
 'LDA_raw':{'kind':'lda'}, 'FBCSP_LDA':{'kind':'fbcsp_lda'},
 'EEGNet':dict(use_nf_eeg=False,use_eegnet=True,use_multiscale=False,use_transformer=False,use_bilstm=False),
 'DeepCNN':dict(use_nf_eeg=False,use_eegnet=False,use_multiscale=True,use_transformer=False,use_bilstm=False),
 'NF_EEG':dict(use_nf_eeg=True,use_eegnet=False,use_multiscale=True,use_transformer=False,use_bilstm=False),
 'ConvoReleNet':dict(use_nf_eeg=False,use_eegnet=False,use_multiscale=True,use_transformer=True,use_bilstm=True),
 'NF_EEG_Transformer':dict(use_nf_eeg=True,use_eegnet=False,use_multiscale=True,use_transformer=True,use_bilstm=False),
 'NF_EEG_Transformer_BiLSTM':dict(use_nf_eeg=True,use_eegnet=False,use_multiscale=True,use_transformer=True,use_bilstm=True),
 'NF_EEG_EEGNet':dict(use_nf_eeg=True,use_eegnet=True,use_multiscale=False,use_transformer=False,use_bilstm=False),
 'Full_Hybrid':dict(use_nf_eeg=True,use_eegnet=True,use_multiscale=True,use_transformer=True,use_bilstm=True)
}
def cfg_for_ablation(name):
    base=copy.deepcopy(CFG_DEFAULT)
    for k,v in ABLATIONS[name].items():
        if hasattr(base,k): setattr(base,k,v)
    return base

def run_raw_lda_loso(X,y,subjects):
    rows=[]
    for target in LOSOSUBJECTS:
        tr,te=subject_split_indices(subjects,target); sc=StandardScaler(); a=sc.fit_transform(X[tr].reshape(len(tr),-1))[:,::20]; b=sc.transform(X[te].reshape(len(te),-1))[:,::20]; clf=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'); clf.fit(a,y[tr]); pred=clf.predict(b); rows.append({'subject':target,'accuracy':accuracy_score(y[te],pred),'balanced_accuracy':balanced_accuracy_score(y[te],pred),'macro_f1':f1_score(y[te],pred,average='macro'),'kappa':cohen_kappa_score(y[te],pred)})
    return pd.DataFrame(rows)

def fbcsp_feature_matrix(Xfit,yfit,Xapply):
    a=[]; b=[]
    for lo,hi in FB_BANDS:
        xf=bandpass_trials(Xfit,lo,hi,CFG_DEFAULT.sfreq); xa=bandpass_trials(Xapply,lo,hi,CFG_DEFAULT.sfreq); W=ovr_csp_filters(xf,yfit,4); a.append(np.log(np.asarray([np.var(W@t,axis=1) for t in xf])+1e-8)); b.append(np.log(np.asarray([np.var(W@t,axis=1) for t in xa])+1e-8))
    return np.concatenate(a,axis=1),np.concatenate(b,axis=1)

def run_fbcsp_lda_loso(X,y,subjects):
    rows=[]
    for target in LOSOSUBJECTS:
        tr,te=subject_split_indices(subjects,target); Ftr,Fte=fbcsp_feature_matrix(X[tr],y[tr],X[te]); sc=StandardScaler(); Ftr=sc.fit_transform(Ftr); Fte=sc.transform(Fte); clf=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'); clf.fit(Ftr,y[tr]); pred=clf.predict(Fte); rows.append({'subject':target,'accuracy':accuracy_score(y[te],pred),'balanced_accuracy':balanced_accuracy_score(y[te],pred),'macro_f1':f1_score(y[te],pred,average='macro'),'kappa':cohen_kappa_score(y[te],pred)})
    return pd.DataFrame(rows)


In [38]:

# CELL 28 — STATISTICAL ANALYSIS
def paired_stats(df_a, df_b, metric="accuracy"):
    merged = df_a[["subject", metric]].merge(
        df_b[["subject", metric]], on="subject", suffixes=("_a","_b")
    )
    d = merged[f"{metric}_a"].values - merged[f"{metric}_b"].values
    t = stats.ttest_rel(merged[f"{metric}_a"], merged[f"{metric}_b"], nan_policy="omit")
    dz = d.mean() / (d.std(ddof=1) + 1e-12)
    return {
        "mean_diff": float(d.mean()),
        "t_stat": float(t.statistic),
        "p_value_uncorrected": float(t.pvalue),
        "cohen_dz": float(dz),
        "n": int(len(d))
    }

def holm_bonferroni(pvals):
    order = np.argsort(pvals)
    adj = np.empty(len(pvals))
    m = len(pvals)
    running = 0
    for rank, idx in enumerate(order):
        val = (m-rank) * pvals[idx]
        running = max(running, val)
        adj[idx] = min(running, 1.0)
    return adj


In [39]:

# CELL 29 — PLOTS / INTERPRETABILITY
def plot_confusion(cm, names=CFG_DEFAULT.class_names, title="Confusion matrix"):
    fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    plt.show()

def plot_training_history(history):
    if history is None or len(history) == 0:
        return
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(history["epoch"], history["train_loss"], label="train loss")
    ax.plot(history["epoch"], history["val_loss"], label="val loss")
    ax.legend()
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Training history")
    plt.tight_layout()
    plt.show()

def plot_subject_results(results_df):
    fig, ax = plt.subplots(figsize=(9,4))
    ax.bar(results_df["subject"], results_df["accuracy"]*100)
    ax.axhline(25, linestyle="--", linewidth=1)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Strict LOSO per-subject accuracy")
    plt.tight_layout()
    plt.show()


In [40]:

# CELL 30 — FINAL RESULTS TABLES
def summarize_results(df, name="model"):
    mean, lo, hi = confidence_interval_95(df["accuracy"].values)
    out = pd.DataFrame([{
        "model": name,
        "mean_accuracy_pct": 100*mean,
        "std_accuracy_pct": 100*df["accuracy"].std(ddof=1),
        "ci95_low_pct": 100*lo,
        "ci95_high_pct": 100*hi,
        "median_pct": 100*df["accuracy"].median(),
        "min_subject_pct": 100*df["accuracy"].min(),
        "max_subject_pct": 100*df["accuracy"].max(),
        "mean_balanced_accuracy_pct": 100*df["balanced_accuracy"].mean(),
        "mean_macro_f1_pct": 100*df["macro_f1"].mean(),
        "mean_kappa": df["kappa"].mean(),
        "params_m": df["params_m"].median() if "params_m" in df.columns else np.nan,
        "train_seconds_mean": df["train_seconds"].mean() if "train_seconds" in df.columns else np.nan,
    }])
    return out

# Example:
# final_summary = summarize_results(strict_results_df, "Full_Hybrid")
# display(final_summary)


In [41]:

# CELL 31 — CHECKPOINT SAVING / LOADING
def save_experiment_artifact(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    elif isinstance(obj, dict):
        path.write_text(json.dumps(obj, indent=2, default=str))
    else:
        torch.save(obj, path)

def load_model_checkpoint(path, cfg):
    ckpt = torch.load(path, map_location=DEVICE)
    model = HybridNFEEGModel(cfg).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    norm = Normalizer(cfg.normalize)
    norm.mean_ = ckpt["normalizer_mean"]
    norm.std_ = ckpt["normalizer_std"]
    return model, norm, ckpt


In [44]:

# CELL 32 — FINAL EXPERIMENT RUNNER + ARCHITECTURE SUMMARY
def architecture_summary(cfg: CFG):
    print("""
HYBRID NF-EEG / FBGAN / CONVORELENET DESIGN
============================================
RAW EEG (N,22,1000)
        |
        +--> NF-EEG PatchEmbedding2D --> raw embedding --------+
        |
        +--> Multi-scale CNN (k=3,5,7,11,15) --> embedding ----+--> Gated Fusion --> Head --> 4 classes
        |                 |
        |                 +--> Transformer (pre-norm, 4 heads)
        |                             |
        |                             +--> optional BiLSTM
        |
        +--> EEGNet compact spatial-temporal branch ------------+

TARGET-ADAPTATION ONLY:
target calibration EEG
  -> 10-band FB
  -> OVR CSP
  -> LASSO sparse features
  -> FBGAN generator + D_phi + D_psi
  -> synthetic target EEG
  -> QC
  -> conservative fine-tuning

STRICT LOSO:
held-out subject is invisible to preprocessing fitting,
feature selection, GAN training, augmentation selection,
early stopping, calibration, ensemble weighting and model selection.
""")
    print("Trainable parameters:", f"{model_parameter_count(HybridNFEEGModel(cfg))/1e6:.3f} M")

architecture_summary(CFG_DEFAULT)

# Main run:
strict_results, strict_summary, strict_folds = run_strict_loso(CFG_DEFAULT, X_ALL, y_ALL, SUBJECTS_ALL)
print(strict_summary)
display(summarize_results(strict_results, "Full_Hybrid"))

# Target adaptation must be explicitly enabled:
cfg_ta = copy.deepcopy(CFG_DEFAULT)
cfg_ta.use_target_adaptation = True
ta_results = run_target_adaptation_loso(cfg_ta, X_ALL, y_ALL, SUBJECTS_ALL)



HYBRID NF-EEG / FBGAN / CONVORELENET DESIGN
RAW EEG (N,22,1000)
        |
        +--> NF-EEG PatchEmbedding2D --> raw embedding --------+
        |
        +--> Multi-scale CNN (k=3,5,7,11,15) --> embedding ----+--> Gated Fusion --> Head --> 4 classes
        |                 |
        |                 +--> Transformer (pre-norm, 4 heads)
        |                             |
        |                             +--> optional BiLSTM
        |
        +--> EEGNet compact spatial-temporal branch ------------+

TARGET-ADAPTATION ONLY:
target calibration EEG
  -> 10-band FB
  -> OVR CSP
  -> LASSO sparse features
  -> FBGAN generator + D_phi + D_psi
  -> synthetic target EEG
  -> QC
  -> conservative fine-tuning

STRICT LOSO:
held-out subject is invisible to preprocessing fitting,
feature selection, GAN training, augmentation selection,
early stopping, calibration, ensemble weighting and model selection.

Trainable parameters: 1.526 M


ValueError: Not enough source subjects for subject-level validation.